## 1. Problem statement

This notebook is retained as secondary exploratory material. The reproducible project pipeline lives in `../src`, `../docs`, and `../reports`.

The objective is to classify ECG5000 heartbeat signal patterns into five benchmark classes. This is a research and education prototype, not a clinically validated diagnostic system.


In [ ]:
# Optional notebook setup. From the repository root, prefer:
# pip install -r requirements.txt
# pip install -r requirements-inception.txt


## 2. Dataset

We will use the dataset [ECG5000](https://timeseriesclassification.com/description.php?Dataset=ECG5000) with 7600 training data and 1900 test data.

Each data contains a cardiac cycle with 140 samples, and that can belong to one of 5 categories:

1. Normal
2. Abnormal: premature ventricular contraction
3. Abnormal: premature supraventricular contraction
4. Abnormal: ectopic beat
5. Abnormal: but unknown pathology

![](https://drive.google.com/uc?export=view&id=1x_sUD1rbM4MM4--s9D4wacRIWEo8aAzL)


## 3. Class imbalance

The problem with the ECG5000 set is that it contains 4427 normal data and 3173 abnormal data, that is, it is unbalanced.

In fact, for certain abnormal categories (2 to 5) there are very few data:

| Category    |  Samples|
|-------------|------------|
| 1 (normal)  | 4427       |
| 2 (abnormal) | 2683       |
| 3 (abnormal) | 149        |
| 4 (abnormal) | 306        |
| 5 (abnormal) | 35         |


We can see that the critical case is category 5 with only 35 samples. We need an approach capable of correctly classifying these data. As a first step in our solution approach, we will split our training and validation data sets keeping the proportion of each type of abnormality for each split.




## 4. Exploration and preprocessing

In [ ]:
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "dataset"
df_train = pd.read_csv(DATA_DIR / "ECG5000_train.csv")
df_test = pd.read_csv(DATA_DIR / "ECG5000_test.csv")


In [ ]:
print(df_train.shape)
print(df_test.shape)

In [ ]:
df_train.head()

In [ ]:
df_train['0'].value_counts()

In [ ]:
df_test['0'].value_counts()

### Train/test split

In [ ]:
from fastai.data.transforms import IndexSplitter, ItemGetter
from fastai.learner import ClassificationInterpretation, Learner
from fastai.losses import CrossEntropyLossFlat
from fastai.optimizer import ranger
from fastai.metrics import accuracy
from tsai.data.core import TSTensor
from tsai.data.preprocessing import TSStandardize
from tsai.data.validation import TSTensorBlock
from tsai.models.InceptionTime import InceptionTime
from fastai.data.block import CategoryBlock, DataBlock


In [ ]:
# Pandas a arreglos numpy
datos_train = df_train.values

# Etiquetas
cat_train = datos_train[:,0]



In [ ]:
print(datos_train.shape)
datos_train

In [ ]:
print(cat_train.shape)
cat_train 

In [ ]:
# extracting categories
x_train_1 = datos_train[cat_train==1,1:]
x_train_2 = datos_train[cat_train==2,1:]
x_train_3 = datos_train[cat_train==3,1:]
x_train_4 = datos_train[cat_train==4,1:]
x_train_5 = datos_train[cat_train==5,1:]

In [ ]:
print('x_train_1:', x_train_1.shape)
print('x_train_2:', x_train_2.shape)
print('x_train_3:', x_train_3.shape)
print('x_train_4:', x_train_4.shape)
print('x_train_5:', x_train_5.shape)

In [ ]:
# Visualization
import matplotlib.pyplot as plt
import numpy as np

ind = 10
normal = x_train_1[ind]
anormal_2 = x_train_2[ind]
anormal_3 = x_train_3[ind]
anormal_4 = x_train_4[ind]
anormal_5 = x_train_5[ind]

plt.figure(figsize=(10,8))
plt.grid()
plt.subplot(2,2,1)
plt.plot(np.arange(140), normal)
plt.plot(np.arange(140), anormal_2, 'r--')
plt.subplot(2,2,2)
plt.plot(np.arange(140), normal)
plt.plot(np.arange(140), anormal_3, 'r--')
plt.subplot(2,2,3)
plt.plot(np.arange(140), normal)
plt.plot(np.arange(140), anormal_4, 'r--')
plt.subplot(2,2,4)
plt.plot(np.arange(140), normal)
plt.plot(np.arange(140), anormal_5, 'r--')

## 5. InceptionTime baseline 

In [ ]:

indexes_train_2 = np.random.choice(x_train_2.shape[0], round(0.8*len(x_train_2)), replace=False)
X_train_2 = x_train_2[indexes_train_2, :]

indexes_train_3 = np.random.choice(x_train_3.shape[0], round(0.8*len(x_train_3)), replace=False)
X_train_3 = x_train_3[indexes_train_3, :]

indexes_train_4 = np.random.choice(x_train_4.shape[0], round(0.8*len(x_train_4)), replace=False)
X_train_4 = x_train_4[indexes_train_4, :]

indexes_train_5 = np.random.choice(x_train_5.shape[0], round(0.8*len(x_train_5)), replace=False)
X_train_5 = x_train_5[indexes_train_5, :]

num_samples_1 = len(indexes_train_2) + len(indexes_train_3) + len(indexes_train_4) + len(indexes_train_5)
indexes_train_1 = np.random.choice(x_train_1.shape[0],num_samples_1 , replace=False)
X_train_1 = x_train_1[indexes_train_1, :]


# readding labels
cat_train_1 = np.ones((X_train_1.shape[0], 1))

cat_train_2 =  np.zeros((X_train_2.shape[0],1))
cat_train_2[:] = 2 

cat_train_3 =  np.zeros((X_train_3.shape[0],1))
cat_train_3[:] = 3

cat_train_4 =  np.zeros((X_train_4.shape[0],1))
cat_train_4[:] = 4

cat_train_5 =  np.zeros((X_train_5.shape[0],1))
cat_train_5[:] = 5

Y_train = np.concatenate((cat_train_1, cat_train_2, cat_train_3, cat_train_4, cat_train_5))


In [ ]:
Y_train.shape

In [ ]:
data_train = np.concatenate((X_train_1, X_train_2, X_train_3, X_train_4, X_train_5))
data_train.shape


In [ ]:
indexes_valid_1 = np.setdiff1d(np.arange(x_train_1.shape[0]), indexes_train_1)
X_valid_1 = x_train_1[indexes_valid_1,:]

indexes_valid_2 = np.setdiff1d(np.arange(x_train_2.shape[0]), indexes_train_2)
X_valid_2 = x_train_2[indexes_valid_2,:]

indexes_valid_3 = np.setdiff1d(np.arange(x_train_3.shape[0]), indexes_train_3)
X_valid_3 = x_train_3[indexes_valid_3,:]


indexes_valid_4 = np.setdiff1d(np.arange(x_train_4.shape[0]), indexes_train_4)
X_valid_4 = x_train_4[indexes_valid_4,:]

indexes_valid_5 = np.setdiff1d(np.arange(x_train_5.shape[0]), indexes_train_5)
X_valid_5 = x_train_5[indexes_valid_5,:]


#readding labels
cat_valid_1 = np.ones((X_valid_1.shape[0], 1))

cat_valid_2 =  np.zeros((X_valid_2.shape[0],1))
cat_valid_2[:] = 2

cat_valid_3 =  np.zeros((X_valid_3.shape[0],1))
cat_valid_3[:] = 3

cat_valid_4 =  np.zeros((X_valid_4.shape[0],1))
cat_valid_4[:] = 4

cat_valid_5 =  np.zeros((X_valid_5.shape[0],1))
cat_valid_5[:] = 5

Y_valid = np.concatenate((cat_valid_1,  cat_valid_2, cat_valid_3,cat_valid_4,cat_valid_5))
Y_valid.shape


In [ ]:
data_valid = np.concatenate((X_valid_1, X_valid_2, X_valid_3, X_valid_4, X_valid_5))
data_valid.shape

In [ ]:
Data_train_1 = np.concatenate((data_train, data_valid))
Target_train_1 = np.concatenate((Y_train, Y_valid))

In [ ]:
Data_train_1.shape

In [ ]:
Target_train_1.shape

In [ ]:
Target_train_1 = np.squeeze(Target_train_1, axis=1)
Target_train_1.shape

### Preprocessing and normalization

In [ ]:
from sklearn.preprocessing import MinMaxScaler
min_max_scaler = MinMaxScaler()

Data_train_1_s = min_max_scaler.fit_transform(Data_train_1)
print('Mínimo y máximo originales: {:.1f}, {:.1f}'.format(np.min(Data_train_1), np.max(Data_train_1)))
print('Mínimo y máximo normalización: {:.1f}, {:.1f}'.format(np.min(Data_train_1_s), np.max(Data_train_1_s)))



In [ ]:
# adding extra dimensions
Data_train_1_s = np.reshape(Data_train_1_s, (Data_train_1_s.shape[0], Data_train_1_s.shape[1],1))


In [ ]:
Data_train_1_s=Data_train_1_s.transpose(0, 2, 1)
Data_train_1_s.shape

In [ ]:
Target_train_1 = Target_train_1.astype('int')

Target_train_1 = Target_train_1.astype('str')

In [ ]:
np.save('X.npy', Data_train_1_s)
np.save('y.npy', Target_train_1)

In [ ]:
X = np.load('X.npy', mmap_mode='r')
y = np.load('y.npy', mmap_mode='r')

In [ ]:
print('X shape:', X.shape)
print('y shape:', y.shape)
print('X type:',type(X))
print('y type:',type(y))

In [ ]:
# Dataset split function
splits = (L(np.arange(0, len(Y_train)), use_list=True),
          L(np.arange(len(Y_train), len(X)), use_list=True))
splits


Since we use memmap, the data is read directly from memory. Now, to create and use your own data, it must be in a three-dimensional array formatted as:
* Samples
* Variables
* Length (or timesteps)

To use this we have a special `TSTensor` built to handle such data:

In [ ]:
t = TSTensor(X)
t

In [ ]:
# Creating our datablock
splitter = IndexSplitter(splits[1])
getters = [ItemGetter(0), ItemGetter(1)]
dblock = DataBlock(blocks=(TSTensorBlock, CategoryBlock),
                   getters=getters,
                   splitter=splitter)

In [ ]:
src = itemify(X, y)
print(len(src))
print('first element:',src[0])

In [ ]:
dls = dblock.dataloaders(src, bs=64, val_bs=128)

dls.show_batch(max_n=3)

In [ ]:
dls.vocab

### InceptionTime architecture
The particular architecture we are using is the [Inception Time](https://towardsdatascience.com/deep-learning-for-time-series-classification-inceptiontime-245703f422db). 
To do this we need the number of input classes and our number of variables:

The line of code "net = InceptionTime(inp_vars, dls.c)" instantiates a neural network model called "InceptionTime" and assigns it to the variable "net".

The "InceptionTime" class is an implementation of a convolutional neural network (CNN) architecture for time series, which uses convolutional filters of different sizes and concatenation operations to extract features from the input time series. This class is often used for time series regression or classification problems.

The arguments passed to the "InceptionTime" class are "inp_vars" and "dls.c". "inp_vars" is the number of features or variables in the input data, which is used to define the number of input channels in the initial convolution layer of the neural network. "dls.c" is the number of classes or categories in the classification problem (if any), which is used to define the number of neurons in the output layer of the neural network.

### Model 1: InceptionTime

In [ ]:
inp_vars = dls.dataset[0][0].shape[-2]
net = InceptionTime(inp_vars, dls.c)

In [ ]:
learn_1 = Learner(dls, net, loss_func=CrossEntropyLossFlat(), metrics=accuracy, opt_func=ranger)

In [ ]:
learn_1.lr_find()

In [ ]:
lr = 3e-03
learn_1.fit_flat_cos(10, lr)

In [ ]:
# getting the confusion matrix
interp = ClassificationInterpretation.from_learner(learn_1)
interp.plot_confusion_matrix()

## 6. EasyEnsemble baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from imblearn.ensemble import EasyEnsembleClassifier

RF estimator

In [ ]:
# Creation of the base classifier
rf = RandomForestClassifier(n_estimators=100, random_state=10)

# Creation of our clasificador EasyEnsemble
eec = EasyEnsembleClassifier(estimator=rf, n_estimators=100, random_state=10)

In [ ]:
Data_train_1_s2 = Data_train_1_s[:,0,:]
Data_train_1_s2.shape

In [ ]:
# trainning EasyEnsembleClassifier
eec.fit(Data_train_1[0:len(Y_train)-1,:], Target_train_1[0:len(Y_train)-1])

In [ ]:
# Predict the values of the class in the test set
y_pred = eec.predict(Data_train_1[len(Y_train)::, :])

# Model performance evaluation
acc = accuracy_score(Target_train_1[len(Y_train)::], y_pred)
acc

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


class_names = ['class 1', 'class 2', 'class 3', 'class 4', 'class 5']


y_target = Target_train_1[len(Y_train)::].astype('int')
y_pred = y_pred.astype('int')


cm = confusion_matrix(y_target, y_pred)



fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
plt.xlabel('Predictions')
plt.ylabel('Ground truth')
plt.show()

SVM estimator

In [ ]:
# Create an SVM estimator
from sklearn.svm import SVC
svm = SVC(kernel='poly', random_state=10)

# Create an EasyEnsemble classifier with the SVM estimator
ee_svm = EasyEnsembleClassifier(estimator=svm, n_estimators=100, random_state=10)

In [ ]:
# Training the EasyEnsemble classifier
ee_svm.fit(Data_train_1[0:len(Y_train)-1,:], Target_train_1[0:len(Y_train)-1])

In [ ]:
# Predict the values of the class in the test set
y_pred = ee_svm.predict(Data_train_1[len(Y_train)::, :])

# Model performance evaluation
acc = accuracy_score(Target_train_1[len(Y_train)::], y_pred)
acc

In [ ]:

# Define the class labels
class_names = ['class 1', 'class 2', 'class 3', 'class 4', 'class 5']


y_target = Target_train_1[len(Y_train)::].astype('int')
y_pred = y_pred.astype('int')


cm = confusion_matrix(y_target, y_pred)




fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
plt.xlabel('Predictions')
plt.ylabel('Ground truth')
plt.show()

## 7. SVM EasyEnsemble baseline

In [ ]:

ee_svm.fit(Data_train_1[0:len(Y_train)-1,:], Target_train_1[0:len(Y_train)-1])

In [ ]:

y_pred = ee_svm.predict(Data_train_1[len(Y_train)::, :])

acc = accuracy_score(Target_train_1[len(Y_train)::], y_pred)
acc

In [ ]:


class_names = ['class 1', 'class 2', 'class 3', 'class 4', 'class 5']


y_target = Target_train_1[len(Y_train)::].astype('int')
y_pred = y_pred.astype('int')

cm = confusion_matrix(y_target, y_pred)




fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=ax)
plt.xlabel('Predictions')
plt.ylabel('Ground truth')
plt.show()

## 8. Final comparison and conclusion

| Model | Test accuracy | Macro-F1 | Balanced accuracy | Role |
|---|---:|---:|---:|---|
| XGBoost | 0.9863 | 0.9092 | 0.8840 | Strongest individual supervised model |
| InceptionTime | 0.9021 | 0.6078 | 0.7509 | Neural time-series baseline |
| Isolation Forest | 0.9363 | 0.9353 | 0.9440 | Binary normal-vs-anomaly detector |
| Ensemble | 0.9853 | 0.8971 | 0.8834 | Final transparent three-model formula |

The current project report prioritizes macro-F1, balanced accuracy, and per-class recall because ECG5000 is imbalanced. The final repository documentation and scripts should be used for reproducible results.
